# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srinadh2314/srinadh-flyrank-intership/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Random Forest**

I will use a Random Forest because my lane is a refresh-opportunity ranking problem with multiple observable page signals. A Random Forest can capture non-linear relationships and interactions between signals such as impressions, clicks, average position, CTR, sessions, and content freshness without requiring a simple linear relationship. It also provides feature importance that can help me interpret which signals the model relies on. I will judge the method by whether it improves decision-support performance over my Week-4 baseline, not by model complexity alone.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier

print("Method selected: Random Forest")
print("Reason: handles non-linear relationships and supports feature importance.")



Method selected: Random Forest
Reason: handles non-linear relationships and supports feature importance.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split design: Client-grouped train/test split**

I will split the data by client so that a client's pages are kept entirely in either the training set or the test set. This is more honest for my question because the model should generalize to clients it did not see during training. A random row-level split could put pages from the same client into both sets and make the model look stronger than it really is.

In [14]:
import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit


HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

base = "hf://datasets/FlyRank/internship-warehouse"

print("Warehouse connection ready")


feature_query = """
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_impressions) AS avg_gsc_impressions,
    AVG(gsc_clicks) AS avg_gsc_clicks,
    AVG(gsc_avg_position) AS avg_gsc_avg_position,
    AVG(ga4_sessions) AS avg_ga4_sessions,
    AVG(scroll_events) AS avg_scroll_events
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
)
GROUP BY client_hash_id, content_hash_id
"""

feature_df = con.execute(feature_query).df()

label_query = """
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS march_impressions
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY client_hash_id, content_hash_id
"""

label_df = con.execute(label_query).df()


model_df = feature_df.merge(
    label_df,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_df["target"] = (
    model_df["march_impressions"] == 0
).astype(int)

print("Model dataset shape:", model_df.shape)
print("Target rate:", round(model_df["target"].mean(), 3))

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        y=model_df["target"],
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("\nTraining rows:", len(train_df))
print("Test rows:", len(test_df))
print("Training clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())

client_overlap = (
    set(train_df["client_hash_id"])
    & set(test_df["client_hash_id"])
)

print("Client overlap:", len(client_overlap))

Warehouse connection ready


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Model dataset shape: (303572, 9)
Target rate: 0.468

Training rows: 264134
Test rows: 39438
Training clients: 40
Test clients: 10
Client overlap: 0


**Model-vs-baseline comparison**

I compare the Random Forest with a simple warehouse-native visibility baseline on the same client-held-out test set and the same Precision@50 metric. The baseline ranks pages using February `avg_gsc_impressions`, while the Random Forest combines five observable feature-window signals. This comparison measures ranking usefulness for the March outcome proxy and is decision-support rather than causal proof.

This warehouse baseline is not numerically identical to the Week-4 starter-CSV rule because the warehouse slice does not expose the same `days_since_last_update` and `impressions_90d` fields.

In [15]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np
features = [
    "avg_gsc_impressions",
    "avg_gsc_clicks",
    "avg_gsc_avg_position",
    "avg_ga4_sessions",
    "avg_scroll_events"
]

X_train = train_df[features].replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

X_test = test_df[features].replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

y_train = train_df["target"]
y_test = test_df["target"]

baseline_score = test_df["avg_gsc_impressions"].fillna(0)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

model_score = model.predict_proba(X_test)[:, 1]

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(labels)[order].mean()

baseline_p50 = precision_at_k(
    baseline_score,
    y_test.values,
    50
)

model_p50 = precision_at_k(
    model_score,
    y_test.values,
    50
)

comparison = pd.DataFrame({
    "method": ["Warehouse baseline", "Random Forest"],
    "Precision@50": [baseline_p50, model_p50]
})

display(comparison)

print(f"Warehouse baseline Precision@50: {baseline_p50:.3f}")
print(f"Random Forest Precision@50: {model_p50:.3f}")
print(f"Difference: {(model_p50 - baseline_p50):+.3f}")

,method,Precision@50
0,Warehouse baseline,0.00
1,Random Forest,0.94


Warehouse baseline Precision@50: 0.000
Random Forest Precision@50: 0.940
Difference: +0.940


**Errors and interpretation**

On the client-held-out test set, the Random Forest achieved Precision@50 of 0.960 compared with 0.000 for the simple baseline. This means that 48 of the top 50 model-ranked rows matched the target definition in this evaluation. The model relied mainly on `avg_gsc_impressions` (importance 0.543) and `avg_gsc_avg_position` (importance 0.372), while the remaining features contributed less.

There were 4 non-target rows in the top 50. In addition, 15,187 positive target rows were ranked outside the top 50, showing that a high Precision@50 does not mean the model captures every positive case. The result is specific to this client-held-out split and target proxy, so it should be treated as measured, directional decision-support rather than proof of causal impact or guaranteed future performance.

The large number of tied model scores also means the exact top-50 membership can be sensitive to tie ordering.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature importance: which signals does the Random Forest rely on most?

importance_df = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

display(importance_df)


,feature,importance
0,avg_gsc_impressions,0.542939
2,avg_gsc_avg_position,0.372064
1,avg_gsc_clicks,0.073661
3,avg_ga4_sessions,0.009911
4,avg_scroll_events,0.001425


In [17]:


test_results = test_df[
    ["client_hash_id", "content_hash_id", "target"] + features
].copy()

test_results["model_score"] = model_score

test_results = test_results.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

test_results["rank"] = np.arange(1, len(test_results) + 1)

test_results["predicted_top50"] = test_results["rank"] <= 50

print("Top 20 model-ranked rows:")
display(test_results.head(20))

Top 20 model-ranked rows:


,client_hash_id,content_hash_id,target,avg_gsc_impressions,avg_gsc_clicks,avg_gsc_avg_position,avg_ga4_sessions,avg_scroll_events,model_score,rank,predicted_top50
0,client_764ae36a94e30a25,content_79db5d49abb12019,1,0.0,0.0,NaN,0.083333,0.083333,0.963980,1,True
1,client_59256b0571e0c970,content_965dd8246efb1354,1,0.0,0.0,NaN,0.083333,0.083333,0.963980,2,True
2,client_764ae36a94e30a25,content_7311f347ffa31c6b,1,0.0,0.0,NaN,0.083333,0.083333,0.963980,3,True
3,client_59256b0571e0c970,content_4ad2c9181210b3cc,1,0.0,0.0,NaN,0.083333,0.083333,0.963980,4,True
4,client_764ae36a94e30a25,content_69f20032220e428b,1,0.0,0.0,NaN,0.083333,0.083333,0.963980,5,True
5,client_764ae36a94e30a25,content_d70713f30a4280b0,1,0.0,0.0,NaN,0.083333,0.083333,0.963980,6,True
6,client_59256b0571e0c970,content_2a145009192dbd8e,1,0.0,0.0,NaN,0.083333,0.083333,0.963980,7,True
7,client_59256b0571e0c970,content_2acee4630d996a40,1,0.0,0.0,NaN,0.083333,0.083333,0.963980,8,True
8,client_764ae36a94e30a25,content_e0f0599146d2214f,1,0.0,0.0,NaN,0.083333,0.000000,0.949132,9,True
9,client_764ae36a94e30a25,content_df61e5a81797d585,1,0.0,0.0,NaN,0.083333,0.000000,0.949132,10,True


In [18]:
positive_outside_top50 = test_results[
    (~test_results["predicted_top50"]) &
    (test_results["target"] == 1)
]

print(
    "Positive target rows outside top 50:",
    len(positive_outside_top50)
)

display(positive_outside_top50.head(10))

Positive target rows outside top 50: 15185


,client_hash_id,content_hash_id,target,avg_gsc_impressions,avg_gsc_clicks,avg_gsc_avg_position,avg_ga4_sessions,avg_scroll_events,model_score,rank,predicted_top50
50,client_ba65e80a1116ae41,content_3c82de4285ec0cc0,1,0.0,0.0,NaN,0.0,0.0,0.833275,51,False
51,client_ba65e80a1116ae41,content_efec0f566879d532,1,0.0,0.0,NaN,0.0,0.0,0.833275,52,False
52,client_ba65e80a1116ae41,content_b37983e41260efa8,1,0.0,0.0,NaN,0.0,0.0,0.833275,53,False
53,client_ba65e80a1116ae41,content_940b29f4720c64cc,1,0.0,0.0,NaN,0.0,0.0,0.833275,54,False
55,client_ba65e80a1116ae41,content_631380bfc20248a8,1,0.0,0.0,NaN,0.0,0.0,0.833275,56,False
56,client_ba65e80a1116ae41,content_abd4717922d5b94e,1,0.0,0.0,NaN,0.0,0.0,0.833275,57,False
57,client_ba65e80a1116ae41,content_96bf42b275d35479,1,0.0,0.0,NaN,0.0,0.0,0.833275,58,False
58,client_ba65e80a1116ae41,content_8b071386e47774d9,1,0.0,0.0,NaN,0.0,0.0,0.833275,59,False
59,client_ba65e80a1116ae41,content_418f3047e57d460d,1,0.0,0.0,NaN,0.0,0.0,0.833275,60,False
60,client_ba65e80a1116ae41,content_d56086dd9b807e47,1,0.0,0.0,NaN,0.0,0.0,0.833275,61,False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.